CEE Government Bond Yield Correlation Analysis
Hermann Ossani

Data source: FRED

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from fredapi import Fred
from scipy import stats

In [ ]:
FRED_API_KEY = "779ebed3d62c58406e3855f082a47187"
fred = Fred(api_key=FRED_API_KEY)

In [ ]:
SERIES = {                                   # Initialise a dictionary with country names and their corresponding FRED series IDs
    "Poland":       "IRLTLT01PLM156N",
    "Czech Rep.":   "IRLTLT01CZM156N",
    "Hungary":      "IRLTLT01HUM156N",
    "Slovakia":     "IRLTLT01SKM156N",
    "Austria":      "IRLTLT01ATM156N",
    "Bulgaria":     "IRLTLT01BGM156N",
    "Croatia":      "IRLTLT01HRM156N",
    "Romania":      "IRLTLT01ROM156N",
    "Germany":      "IRLTLT01DEM156N",
}

print("Fetching data from FRED...\n")

raw = {}                                    # Create an empty dictionary to store the raw data for each country
for country, series_id in SERIES.items():   # Loop through each country and its corresponding FRED series
    try:        # Attempt to fetch the data series from FRED, starting from January 2005
        s = fred.get_series(series_id, observation_start="2005-01-01")   # This line of code is
        raw[country] = s
        print(f"  ✓  {country:<14} {len(s)} months  "
              f"({s.first_valid_index().strftime('%b %Y')} – "
              f"{s.last_valid_index().strftime('%b %Y')})")
    except Exception as e:
        print(f"  ✗  {country:<14} Could not load: {e}")

df = pd.DataFrame(raw)
df_clean = df.dropna()

print(f"\nClean dataset : {len(df_clean)} months of overlapping data")
print(f"Period        : {df_clean.index[0].strftime('%B %Y')} — "
      f"{df_clean.index[-1].strftime('%B %Y')}")

df_changes = df_clean.diff().dropna() * 100

corr = df_changes.corr()

COUNTRY_COLORS = {
    "Poland":     "#E63946",
    "Czech Rep.": "#457B9D",
    "Hungary":    "#2A9D8F",
    "Slovakia":   "#E9C46A",
    "Austria":    "#264653",
    "Bulgaria":   "#F4A261",
    "Croatia":    "#A8DADC",
    "Romania":    "#6A0572",
    "Germany":    "#888888",
}

# Figure 1 — Heatmap + Time Series
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor("#F8F8F8")
for ax in axes:
    ax.set_facecolor("#F8F8F8")

fig.suptitle(
    "CEE Government Bond Yield Analysis  |  10-Year Benchmarks  |  2005–Present",
    fontsize=14, fontweight="bold", y=1.01, color="#1A1A2E"
)

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, ax=axes[0], mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=-1, vmax=1, linewidths=0.8, linecolor="white",
    annot_kws={"size": 10, "weight": "bold"},
    cbar_kws={"label": "Pearson r", "shrink": 0.75}
)
axes[0].set_title("Correlation of Monthly Yield Changes (bps)",
    fontsize=11, pad=12, color="#1A1A2E")
axes[0].tick_params(axis="x", rotation=35, labelsize=9)
axes[0].tick_params(axis="y", rotation=0,  labelsize=9)

for col in df_clean.columns:
    lw = 2.2 if col == "Germany" else 1.4
    ls = "--" if col == "Germany" else "-"
    axes[1].plot(df_clean.index, df_clean[col], label=col,
                 color=COUNTRY_COLORS[col], linewidth=lw, linestyle=ls)

events = [
    ("2008-09-01", "2009-06-01", "#FF6B6B", "GFC"),
    ("2020-02-01", "2020-06-01", "#FFA500", "COVID"),
    ("2022-01-01", "2023-09-01", "#9B59B6", "Rate hike cycle"),
]
for start, end, color, label in events:
    axes[1].axvspan(start, end, alpha=0.10, color=color)

axes[1].set_title("10-Year Government Bond Yields",
    fontsize=11, pad=12, color="#1A1A2E")
axes[1].set_ylabel("Yield (%)", fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
axes[1].legend(frameon=False, fontsize=8, ncol=2)
axes[1].grid(axis="y", alpha=0.25)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

for start, end, color, label in events:
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    axes[1].text(mid, axes[1].get_ylim()[1] * 0.97,
                 label, fontsize=7, color=color, ha="center", alpha=0.9)

plt.tight_layout()
plt.savefig("cee_bond_heatmap.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✓ Saved: cee_bond_heatmap.png")

# Figure 2 — Rolling Correlations vs Germany
fig2, axes2 = plt.subplots(2, 4, figsize=(18, 8), sharey=True)
fig2.patch.set_facecolor("#F8F8F8")
fig2.suptitle(
    "Rolling 24-Month Correlation with German Bund  |  Each CEE Country",
    fontsize=13, fontweight="bold", y=1.01, color="#1A1A2E"
)

cee_countries = [c for c in df_clean.columns if c != "Germany"]

for i, country in enumerate(cee_countries):
    ax = axes2.flatten()[i]
    ax.set_facecolor("#F8F8F8")

    rolling = (df_changes[country]
               .rolling(24)
               .corr(df_changes["Germany"])
               .dropna())
    mean_r = rolling.mean()
    color  = COUNTRY_COLORS[country]

    ax.plot(rolling.index, rolling, color=color, linewidth=1.6)
    ax.axhline(mean_r, color="gray", linestyle="--", linewidth=0.9)
    ax.fill_between(rolling.index, rolling, mean_r,
                    where=(rolling >= mean_r), alpha=0.15, color=color)
    ax.fill_between(rolling.index, rolling, mean_r,
                    where=(rolling <  mean_r), alpha=0.08, color="gray")
    ax.axvspan("2022-01-01", "2023-09-01", alpha=0.07, color="#9B59B6")

    ax.set_title(f"{country}  (avg r = {mean_r:.2f})",
                 fontsize=10, color="#1A1A2E", pad=6)
    ax.set_ylim(-0.5, 1.1)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.3)
    ax.grid(axis="y", alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=7)
    ax.set_ylabel("Pearson r" if i % 4 == 0 else "", fontsize=8)

plt.tight_layout()
plt.savefig("cee_rolling_correlation.png", dpi=150, bbox_inches="tight",
            facecolor=fig2.get_facecolor())
plt.show()
print("✓ Saved: cee_rolling_correlation.png")

# Printed Summary
print("\n" + "=" * 58)
print("KEY FINDINGS")
print("=" * 58)

print(f"\n{'Country':<14} {'Avg Yield':>10} {'Std Dev':>10} {'Corr vs DE':>12}")
print("-" * 50)
for country in cee_countries:
    avg  = df_clean[country].mean()
    std  = df_clean[country].std()
    r_de = corr.loc[country, "Germany"]
    print(f"{country:<14} {avg:>9.2f}%  {std:>9.2f}%  {r_de:>11.3f}")

print("\nStrongest correlated pairs:")
pairs = []
for i in range(len(cee_countries)):
    for j in range(i + 1, len(cee_countries)):
        a, b = cee_countries[i], cee_countries[j]
        pairs.append((a, b, corr.loc[a, b]))
pairs.sort(key=lambda x: x[2], reverse=True)
for a, b, r in pairs[:5]:
    print(f"  {a} ↔ {b:<14}  r = {r:.3f}")

print("\nWeakest correlated pairs:")
for a, b, r in pairs[-3:]:
    print(f"  {a} ↔ {b:<14}  r = {r:.3f}")

print("\nExcluded countries and reason:")
excluded = {
    "Russia":         "Data disrupted post-2022 sanctions",
    "Ukraine":        "Sovereign bond market disrupted by war",
    "Belarus":        "No liquid sovereign bond market",
    "Kosovo":         "No sovereign bond market",
    "Albania":        "Not available on FRED",
    "Bosnia & Herz.": "Not available on FRED",
    "Serbia":         "Not available on FRED",
    "Moldova":        "Not available on FRED",
}
for country, reason in excluded.items():
    print(f"  {country:<20} {reason}")

print("=" * 58)